# Methodological comparison across variants

Reads `results/<variant>/{real,null1,null2}/metrics.json` and produces the decision-surface table + figures consolidating across variants.

Run each variant first (from its worktree or from main):
```bash
python -m pls_explorer.runner --config configs/<name>.yaml
```

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pls_explorer.config import RESULTS_DIR

In [ ]:
rows = []
for variant_dir in sorted(RESULTS_DIR.iterdir()):
    if not variant_dir.is_dir():
        continue
    for label in ['real', 'null1', 'null2']:
        mfile = variant_dir / label / 'metrics.json'
        if not mfile.exists():
            continue
        with open(mfile) as f:
            m = json.load(f)
        rows.append({
            'variant': variant_dir.name,
            'label': label,
            'n_comp': m['n_components'],
            'q2': m['q2_global'],
            'rmsecv': m['rmsecv'],
            'rmsecv_over_rmsec': m['rmsecv_over_rmsec'],
            'diagonality': m['w_structure']['diagonality_index'],
            'off_diag_entropy': m['w_structure']['off_diagonal_entropy'],
            'perm_p': m.get('permutation_p_value', np.nan),
        })
df = pd.DataFrame(rows)
df

## Decision-surface table

Pivot to show variants × {real, null1, null2} for each key metric.

In [ ]:
for metric in ['q2', 'rmsecv', 'diagonality', 'off_diag_entropy']:
    print(f'\n=== {metric} ===')
    print(df.pivot(index='variant', columns='label', values=metric).round(3))

## Real-vs-nulls discrimination

Which variant gives the cleanest separation between real and each null?

In [ ]:
wide = df.pivot(index='variant', columns='label', values='diagonality')
wide['real_minus_null1'] = wide['real'] - wide['null1']
wide['real_minus_null2'] = wide['real'] - wide['null2']
wide['span'] = wide[['real', 'null1', 'null2']].max(axis=1) - wide[['real', 'null1', 'null2']].min(axis=1)
wide.sort_values('span', ascending=False)